In [1]:
import json
import random
import numpy as np
from logic import recommend_courses
from tqdm import tqdm
import spacy
nlp = spacy.load("pl_core_news_sm")


In [5]:
with open("coursedata 1.json", "r", encoding="utf-8") as f:
    all_courses = json.load(f)

random.shuffle(all_courses)
test_courses = all_courses[:2000]  

def generate_query(course):
    return {
        "search_topic": course["course_name"],
        "search_skills": " ".join(course["course_goals"]),
        "search_level": "średniozaawansowany"
    }

precision_scores = []
recall_scores = []
accuracy_scores = []
mrr_scores = []
dcg_scores = []
ndcg_scores = []

def dcg(relevances):
    """Oblicza DCG"""
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevances))

def ndcg(relevances):
    """Oblicza NDCG"""
    ideal_relevances = sorted(relevances, reverse=True)
    return dcg(relevances) / dcg(ideal_relevances) if dcg(ideal_relevances) > 0 else 0

for course in tqdm(test_courses, desc="Testowanie modelu", unit="kurs"):
    query = generate_query(course)
    expected_course_name = course["course_name"]

    recommended_courses = recommend_courses(query["search_topic"], query["search_skills"], query["search_level"])
    recommended_names = [c[0] for c in recommended_courses]

    relevant_recommendations = int(expected_course_name in recommended_names)
    precision = relevant_recommendations / len(recommended_names) if recommended_names else 0
    recall = relevant_recommendations / 1  

    precision_scores.append(precision)
    recall_scores.append(recall)

    accuracy_scores.append(relevant_recommendations)

    if expected_course_name in recommended_names:
        rank = recommended_names.index(expected_course_name) + 1
        mrr_scores.append(1 / rank)
    else:
        mrr_scores.append(0)

    relevances = [1 if name == expected_course_name else 0 for name in recommended_names]
    dcg_scores.append(dcg(relevances))
    ndcg_scores.append(ndcg(relevances))

avg_precision = np.mean(precision_scores)
avg_recall = np.mean(recall_scores)
f1_score = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
accuracy = np.mean(accuracy_scores)
mrr = np.mean(mrr_scores)
ndcg_score = np.mean(ndcg_scores)

print(f"Średnia precyzja (Precision): {avg_precision:.2f}")
print(f"Średni recall: {avg_recall:.2f}")
print(f"F1-score: {f1_score:.2f}")
print(f"Dokładność (Accuracy): {accuracy:.2f}")
print(f"Mean Reciprocal Rank (MRR): {mrr:.2f}")
print(f"NDCG: {ndcg_score:.2f}")

Testowanie modelu: 100%|██████████| 2000/2000 [32:47<00:00,  1.02kurs/s]

Średnia precyzja (Precision): 0.29
Średni recall: 0.87
F1-score: 0.44
Dokładność (Accuracy): 0.87
Mean Reciprocal Rank (MRR): 0.80
NDCG: 0.82
